In [ ]:
!pip install keras_rs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.6/94.6 kB 3.2 MB/s eta 0:00:00


In [ ]:
import pprint
import os
from collections import defaultdict
os.environ["KERAS_BACKEND"] = "jax"
import numpy as np
import pandas as pd
import tensorflow as tf
import keras
import keras_hub
import tensorflow as tf
from keras import ops
import keras_rs
import itertools
import gc
from sklearn.metrics.pairwise import cosine_similarity

2026-04-28 14:20:00.346334: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777386000.837949      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777386000.946868      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777386002.020712      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777386002.020759      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777386002.020762      23 computation_placer.cc:177] computation placer alr

In [ ]:
PAD_ITEM_ID = 0          
MAX_CONTEXT_LENGTH = 10  
BATCH_SIZE = 128
K_TEST = [10, 20]
DROPOUT = 0.1
NUM_EPOCHS = 200

In [ ]:
hidden_dims = [256, 512, 1024]
num_layers_list = [1, 2, 3]
num_heads_list = [1, 2, 3]
learning_rates = [1e-4, 1e-5]
trainable = [True, False]
models_to_test = ["word2vec", "bert_multi", 'qwen']

all_combinations = list(itertools.product(hidden_dims, num_layers_list, num_heads_list, learning_rates, models_to_test, trainable))
print(f"Total kombinasi yang akan dijalankan: {len(all_combinations)}")

grid_results = []
csv_filename = "grid_search_results_live.csv"

Total kombinasi yang akan dijalankan: 324


# Data

In [ ]:
data_path = '/kaggle/input/datasets/rahayukartikasari/university-library-data/'

In [ ]:
books = pd.read_csv(f"{data_path}books_enriched.csv", low_memory=False, delimiter=';')
books.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14502 entries, 0 to 14501
Data columns (total 21 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   kode_buku                 14502 non-null  object 
 1   isbn_issn                 12794 non-null  object 
 2   judul_buku                14502 non-null  object 
 3   subjudul                  7818 non-null   object 
 4   penulis                   13490 non-null  object 
 5   nama_belakang             13838 non-null  object 
 6   topik                     13806 non-null  object 
 7   deskripsi                 14502 non-null  object 
 8   bahasa                    14502 non-null  object 
 9   kategori                  14502 non-null  object 
 10  tahun_terbit              14305 non-null  float64
 11  key                       14502 non-null  object 
 12  confidence                14257 non-null  float64
 13  volume_info               13681 non-null  object 
 14  isbn_1

In [ ]:
books[['book_id','judul_full']].head(5)

,book_id,judul_full
0,B000003,Preparing for the Twenty-first Century
1,B000005,The Illustrator 9 Wow! Book
2,B000006,Crucial Decisions leadership in policymaking a...
3,B000010,Basic Biochemistry
4,B000017,Retail Management a strategic approach


In [ ]:
# make bookId
books['bookId'] = books.index + 1
books[['book_id', 'bookId','judul_full']].head(10)

,book_id,bookId,judul_full
0,B000003,1,Preparing for the Twenty-first Century
1,B000005,2,The Illustrator 9 Wow! Book
2,B000006,3,Crucial Decisions leadership in policymaking a...
3,B000010,4,Basic Biochemistry
4,B000017,5,Retail Management a strategic approach
5,B000019,6,The Capital Budgeting Decision economic analys...
6,B000022,7,Logistical Management a systems integration of...
7,B000025,8,Heat Transfer
8,B000027,9,Principles of Horticulture
9,B000044,10,Elements of Econometrics


In [ ]:
id_to_books = dict(zip(books['bookId'], books['book_id']))
id_to_books[0] = ""

In [ ]:
books_to_id = dict(zip(books['book_id'], books['bookId']))
books_to_id[''] = 0

In [ ]:
users = pd.read_csv(f"{data_path}all_users.csv")
users.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1798 entries, 0 to 1797
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Unnamed: 0    1798 non-null   int64  
 1   ID Anggota    1798 non-null   int64  
 2   prodi_id      1786 non-null   object 
 3   jurusan       1786 non-null   object 
 4   fakultas_id   1786 non-null   float64
 5   fakultas      1786 non-null   object 
 6   jenjang       1786 non-null   object 
 7   Nama Anggota  1798 non-null   object 
 8   role          1798 non-null   object 
 9   angkatan      1774 non-null   float64
 10  userId        1798 non-null   int64  
dtypes: float64(2), int64(3), object(6)
memory usage: 154.6+ KB


In [ ]:
users['userId'] = users.index + 1

In [ ]:
users['ID Anggota'].head(2)

0    196207282021211000
1    196211201991032000
Name: ID Anggota, dtype: int64

In [ ]:
id_to_users = dict(zip(users['userId'], users['ID Anggota']))
id_to_users[0] = ""

In [ ]:
users_to_id = dict(zip(users['ID Anggota'], users['userId']))
users_to_id[''] = 0

In [ ]:
trans_pd = pd.read_csv(f'{data_path}transactions_enriched.csv', low_memory=False, delimiter=';')
trans_pd.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5372 entries, 0 to 5371
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   ID Anggota          5372 non-null   int64 
 1   Nama Anggota        5372 non-null   object
 2   Kode Eksemplar      5372 non-null   int64 
 3   Judul               5372 non-null   object
 4   Tanggal Pinjam      5372 non-null   object
 5   tahun_pinjam        5372 non-null   int64 
 6   bulan_pinjam        5372 non-null   int64 
 7   bulan_tahun_pinjam  5372 non-null   object
 8   book_id             5372 non-null   object
dtypes: int64(4), object(5)
memory usage: 377.8+ KB


In [ ]:
trans_pd['userId'] = trans_pd['ID Anggota'].map(users_to_id)
trans_pd['bookId'] = trans_pd['book_id'].map(books_to_id)

In [ ]:
trans_pd['timestamp'] = pd.to_datetime(trans_pd['Tanggal Pinjam'])
trans_pd.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5372 entries, 0 to 5371
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   ID Anggota          5372 non-null   int64         
 1   Nama Anggota        5372 non-null   object        
 2   Kode Eksemplar      5372 non-null   int64         
 3   Judul               5372 non-null   object        
 4   Tanggal Pinjam      5372 non-null   object        
 5   tahun_pinjam        5372 non-null   int64         
 6   bulan_pinjam        5372 non-null   int64         
 7   bulan_tahun_pinjam  5372 non-null   object        
 8   book_id             5372 non-null   object        
 9   userId              5371 non-null   float64       
 10  bookId              5372 non-null   int64         
 11  timestamp           5372 non-null   datetime64[ns]
dtypes: datetime64[ns](1), float64(1), int64(5), object(5)
memory usage: 503.8+ KB


In [ ]:
trans = trans_pd[['userId', 'bookId', 'timestamp']]

# Sequence

In [ ]:
def build_sequence(transaction_data):
    sequences = defaultdict(list)
    for userId, bookId, timestamp in trans.values:
            sequences[userId].append(
                {
                    "bookId": bookId,
                    "timestamp": timestamp,
                }
            )
    # Sort sequences by timestamp for every user.
    for user_id, context in sequences.items():
        context.sort(key=lambda x: x["timestamp"])
        sequences[user_id] = context

    return sequences

In [ ]:
sequences = build_sequence(trans)

In [ ]:
books_count = books.shape[0]
books_count

14502

In [ ]:
def build_negative_seq(sequences):
    examples = {
        "sequence": [],
        "negative_sequence": [],
    }

    for userId in sequences:
        sequence = [int(d["bookId"]) for d in sequences[userId]]

        # Get negative sequence.
        def random_negative_item_id(low, high, positive_lst):
            sampled = np.random.randint(low=low, high=high)
            while sampled in positive_lst:
                sampled = np.random.randint(low=low, high=high)
            return sampled

        negative_sequence = [
            random_negative_item_id(1, books_count + 1, sequence)
            for _ in range(len(sequence))
        ]

        examples["sequence"].append(np.array(sequence))
        examples["negative_sequence"].append(np.array(negative_sequence))

    examples["sequence"] = tf.ragged.constant(examples["sequence"])
    examples["negative_sequence"] = tf.ragged.constant(examples["negative_sequence"])

    return examples

In [ ]:
examples = build_negative_seq(sequences)
ds = tf.data.Dataset.from_tensor_slices(examples).batch(BATCH_SIZE)

I0000 00:00:1777386045.427638      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1777386045.433676      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [ ]:
from collections import defaultdict

def data_split(sequences):
    user_train, user_valid, user_test = {}, {}, {}
    itemnum = 0

    for user, seq in sequences.items():
        # sort by timestamp
        seq_sorted = sorted(seq, key=lambda x: x["timestamp"])
        book_ids = [int(x["bookId"]) for x in seq_sorted]
        itemnum = max(itemnum, max(book_ids))

        n = len(book_ids)

        if n < 2:
            # Panjang 1: Tidak bisa diuji
            user_train[user] = book_ids
            user_valid[user] = []
            user_test[user] = []
        elif n == 2:
            # Panjang 2:
            user_train[user] = book_ids
            user_valid[user] = []
            user_test[user] = []
        elif n == 3:
            # Panjang 3:
            user_train[user] = book_ids[:-1] 
            user_valid[user] = []
            user_test[user] = book_ids
        else:
            # Panjang > 3
            user_train[user] = book_ids[:-2]
            user_valid[user] = book_ids[:-1]
            user_test[user] = book_ids

    return user_train, user_valid, user_test, itemnum

In [ ]:
train, valid, test, itemnum = data_split(sequences)

In [ ]:
def build_tf_dataset(user_dict, itemnum):
    sequences = []
    negative_sequences = []

    for u, seq in user_dict.items():
        if not seq: continue
        # Ensure sequence is a NumPy array for consistency
        seq_np = np.array(seq, dtype=np.int32)
        # Generate negative samples for the sequence length
        negative_seq_np = np.random.randint(1, itemnum + 1, size=len(seq_np))

        sequences.append(seq_np)
        negative_sequences.append(negative_seq_np)

    # Convert lists of NumPy arrays (sequences) into Ragged Tensors
    ragged_examples = {
        "sequence": tf.ragged.constant(sequences, dtype=tf.int32),
        "negative_sequence": tf.ragged.constant(negative_sequences, dtype=tf.int32),
    }

    # Create the dataset from the ragged tensors, and then batch it
    ds = tf.data.Dataset.from_tensor_slices(ragged_examples)

    return ds

In [ ]:
def _preprocess(example, train=True):
    seq = example["sequence"]
    neg_seq = example["negative_sequence"]
    batch_size = tf.shape(seq)[0]

    if train:
        # Example seq: [A, B, C] -> input: [A, B], target: [B, C]
        inputs = seq[..., :-1]
        targets = seq[..., 1:]
        neg_targets = neg_seq[..., 1:]

        # Truncate to MAX_CONTEXT_LENGTH
        inputs = inputs[..., -MAX_CONTEXT_LENGTH:]
        targets = targets[..., -MAX_CONTEXT_LENGTH:]
        neg_targets = neg_targets[..., -MAX_CONTEXT_LENGTH:]

        # Pad dynamically
        inputs_padded = inputs.to_tensor(shape=[batch_size, MAX_CONTEXT_LENGTH], default_value=PAD_ITEM_ID)
        targets_padded = targets.to_tensor(shape=[batch_size, MAX_CONTEXT_LENGTH], default_value=PAD_ITEM_ID)
        neg_targets_padded = neg_targets.to_tensor(shape=[batch_size, MAX_CONTEXT_LENGTH], default_value=PAD_ITEM_ID)

        # Sample weight: 1.0 for real target items, 0.0 for padding
        sample_weight = tf.cast(targets_padded != PAD_ITEM_ID, dtype="float32")

        return (
            {
                "item_ids": inputs_padded,
                "padding_mask": inputs_padded != PAD_ITEM_ID,
            },
            {
                "positive_sequence": targets_padded,
                "negative_sequence": neg_targets_padded,
            },
            sample_weight
        )
    else:
        # For testing: Context is everything EXCEPT the last item. Target IS the last item.
        # Example seq: [A, B, C] -> context: [A, B], target: [C]
        context = seq[..., :-1]
        target = seq[..., -1:]

        # Truncate context
        context = context[..., -MAX_CONTEXT_LENGTH:]
        
        context_padded = context.to_tensor(shape=[batch_size, MAX_CONTEXT_LENGTH], default_value=PAD_ITEM_ID)
        
        # Flatten target into a 1D tensor [batch_size]
        target_flat = tf.reshape(target.to_tensor(), [batch_size])

        return (
            {
                "item_ids": context_padded,
                "padding_mask": context_padded != PAD_ITEM_ID,
            },
            {
                "positive_sequence": target_flat,
            }
        )

In [ ]:
def preprocess_train(examples):
    return _preprocess(examples, train=True)
def preprocess_val(examples):
    return _preprocess(examples, train=True)
def preprocess_test(examples):
    return _preprocess(examples, train=False)

In [ ]:
user_train, user_valid, user_test, itemnum = data_split(sequences)

In [ ]:
print(f"Jumlah user di Train set: {sum(1 for v in user_train.values() if v)}")
print(f"Jumlah user di Valid set: {sum(1 for v in user_valid.values() if v)}")
print(f"Jumlah user di Test set:  {sum(1 for v in user_test.values() if v)}")

Jumlah user di Train set: 1798
Jumlah user di Valid set: 396
Jumlah user di Test set:  633


In [ ]:
train_ds = build_tf_dataset(user_train, itemnum).batch(BATCH_SIZE).map(preprocess_train)
val_ds = build_tf_dataset(user_valid, itemnum).batch(BATCH_SIZE).map(preprocess_val)
test_ds = build_tf_dataset(user_test, itemnum).batch(BATCH_SIZE).map(preprocess_test)

In [ ]:
for batch in train_ds.take(1):
    print(batch)

({'item_ids': <tf.Tensor: shape=(128, 10), dtype=int32, numpy=
array([[12671, 12671,  9742, ..., 13822, 14415, 10303],
       [    0,     0,     0, ...,     0,     0,     0],
       [    0,     0,     0, ...,     0,     0,     0],
       ...,
       [13486,     0,     0, ...,     0,     0,     0],
       [10185,     0,     0, ...,     0,     0,     0],
       [14306, 10185,     0, ...,     0,     0,     0]], dtype=int32)>, 'padding_mask': <tf.Tensor: shape=(128, 10), dtype=bool, numpy=
array([[ True,  True,  True, ...,  True,  True,  True],
       [False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False],
       ...,
       [ True, False, False, ..., False, False, False],
       [ True, False, False, ..., False, False, False],
       [ True,  True, False, ..., False, False, False]])>}, {'positive_sequence': <tf.Tensor: shape=(128, 10), dtype=int32, numpy=
array([[12671,  9742, 14022, ..., 14415, 10303, 12845],
       [    0,     0,     0, .

In [ ]:
for batch in val_ds.take(1):
    print(batch)

({'item_ids': <tf.Tensor: shape=(128, 10), dtype=int32, numpy=
array([[12671,  9742, 14022, ..., 14415, 10303, 12845],
       [13373, 13373,  9930, ..., 12587, 10107,     0],
       [ 9742, 12563,     0, ...,     0,     0,     0],
       ...,
       [13703, 13711,     0, ...,     0,     0,     0],
       [13045, 13265,     0, ...,     0,     0,     0],
       [ 9752, 13755, 13986, ...,     0,     0,     0]], dtype=int32)>, 'padding_mask': <tf.Tensor: shape=(128, 10), dtype=bool, numpy=
array([[ True,  True,  True, ...,  True,  True,  True],
       [ True,  True,  True, ...,  True,  True, False],
       [ True,  True, False, ..., False, False, False],
       ...,
       [ True,  True, False, ..., False, False, False],
       [ True,  True, False, ..., False, False, False],
       [ True,  True,  True, ..., False, False, False]])>}, {'positive_sequence': <tf.Tensor: shape=(128, 10), dtype=int32, numpy=
array([[ 9742, 14022,  9851, ..., 10303, 12845,  9807],
       [13373,  9930,  9754, .

In [ ]:
testable_seq_lengths = [len(seq) for seq in sequences.values() if len(seq) >= 2]
user_threshold = np.percentile(testable_seq_lengths, 20)

print(f"Ambang Batas (Threshold) Pengguna: {user_threshold} interaksi")

Ambang Batas (Threshold) Pengguna: 2.0 interaksi


In [ ]:
cold_users = set()
warm_users = set()

for user, seq in sequences.items():
    if len(seq) <= user_threshold:
        cold_users.add(user)
    else:
        warm_users.add(user)

print(f"Total Cold Users: {len(cold_users)}")
print(f"Total Warm Users: {len(warm_users)}")

Total Cold Users: 1165
Total Warm Users: 633


In [ ]:
warm_items = set()

# Item yang muncul pada train set adalah warm-item
for user, seq in user_train.items():
    warm_items.update(seq)

# Mencari total seluruh item unik yang ada (berdasarkan dictionary sequence awal)
all_items_in_data = set()
for user, seq in sequences.items():
    for interaksi in seq:
        all_items_in_data.add(int(interaksi["bookId"]))

# Item yang ada di dataset tapi TIDAK ada di train set adalah cold-item
cold_items = all_items_in_data - warm_items

print(f"Total Warm Items: {len(warm_items)}")
print(f"Total Cold Items: {len(cold_items)}")

Total Warm Items: 1747
Total Cold Items: 213


In [ ]:
test_warm_start = {}
test_cold_start = {}

for user, test_seq in user_test.items():
    if not test_seq:
        continue
        
    target_item = test_seq[-1]
    
    if user in warm_users and target_item in warm_items:
        test_warm_start[user] = test_seq
    
    if user in cold_users or target_item in cold_items:
        test_cold_start[user] = test_seq

In [ ]:
print(f"Ukuran Test Set Warm-Start: {len(test_warm_start)} pengguna")
print(f"Ukuran Test Set Cold-Start: {len(test_cold_start)} pengguna")

Ukuran Test Set Warm-Start: 482 pengguna
Ukuran Test Set Cold-Start: 151 pengguna


In [ ]:
ds_test_warm_start = build_tf_dataset(test_warm_start, itemnum).batch(BATCH_SIZE).map(preprocess_test)
for batch in ds_test_warm_start.take(1):
    print("Contoh Batch Warm Test Set:")
    print("Inputs:", batch[0])
    print("Targets:", batch[1])

Contoh Batch Warm Test Set:
Inputs: {'item_ids': <tf.Tensor: shape=(128, 10), dtype=int32, numpy=
array([[ 9742, 14022,  9851, ..., 10303, 12845,  9807],
       [13373, 13373,  9930, ..., 12587, 10107,  9754],
       [14480, 12695,     0, ...,     0,     0,     0],
       ...,
       [12573, 13820, 13754, ..., 13649, 14053, 10271],
       [13980,  1163, 13870, ..., 10110, 13870, 13986],
       [14439, 14439, 13468, ...,     0,     0,     0]], dtype=int32)>, 'padding_mask': <tf.Tensor: shape=(128, 10), dtype=bool, numpy=
array([[ True,  True,  True, ...,  True,  True,  True],
       [ True,  True,  True, ...,  True,  True,  True],
       [ True,  True, False, ..., False, False, False],
       ...,
       [ True,  True,  True, ...,  True,  True,  True],
       [ True,  True,  True, ...,  True,  True,  True],
       [ True,  True,  True, ..., False, False, False]])>}
Targets: {'positive_sequence': <tf.Tensor: shape=(128,), dtype=int32, numpy=
array([13234,  9753, 12695, 14129, 13469,  985

In [ ]:
ds_test_cold_start = build_tf_dataset(test_cold_start, itemnum).batch(BATCH_SIZE).map(preprocess_test)

for batch in ds_test_cold_start.take(1):
    print("Contoh Batch Cold Test Set:")
    print("Inputs:", batch[0])
    print("Targets:", batch[1])

Contoh Batch Cold Test Set:
Inputs: {'item_ids': <tf.Tensor: shape=(128, 10), dtype=int32, numpy=
array([[13639, 14223, 13641, ..., 14080, 12672, 12185],
       [ 3636,  3636,  9870, ...,  1583,  1582,  1582],
       [12348, 10105, 14093, ...,     0,     0,     0],
       ...,
       [14409, 13621, 13395, ...,     0,     0,     0],
       [12200, 14480,     0, ...,     0,     0,     0],
       [12554, 12621,     0, ...,     0,     0,     0]], dtype=int32)>, 'padding_mask': <tf.Tensor: shape=(128, 10), dtype=bool, numpy=
array([[ True,  True,  True, ...,  True,  True,  True],
       [ True,  True,  True, ...,  True,  True,  True],
       [ True,  True,  True, ..., False, False, False],
       ...,
       [ True,  True,  True, ..., False, False, False],
       [ True,  True, False, ..., False, False, False],
       [ True,  True, False, ..., False, False, False]])>}
Targets: {'positive_sequence': <tf.Tensor: shape=(128,), dtype=int32, numpy=
array([13991,  9451,  9875, 14437,  1224, 1445

# Base Model

In [ ]:
class SasRec(keras.Model):
    def __init__(
        self,
        vocabulary_size,
        num_layers,
        num_heads,
        hidden_dim,
        dropout=0.0,
        max_sequence_length=100,
        dtype=None,
        k=20,
        **kwargs,
    ):
        super().__init__(dtype=dtype, **kwargs)

        # ======== Layers ========

        # === Embeddings ===
        self.item_embedding = keras_hub.layers.ReversibleEmbedding(
            input_dim=vocabulary_size,
            output_dim=hidden_dim,
            embeddings_initializer="glorot_uniform",
            embeddings_regularizer=keras.regularizers.l2(0.001),
            dtype=dtype,
            name="item_embedding",
        )
        self.position_embedding = keras_hub.layers.PositionEmbedding(
            initializer="glorot_uniform",
            sequence_length=max_sequence_length,
            dtype=dtype,
            name="position_embedding",
        )
        self.embeddings_add = keras.layers.Add(
            dtype=dtype,
            name="embeddings_add",
        )
        self.embeddings_dropout = keras.layers.Dropout(
            dropout,
            dtype=dtype,
            name="embeddings_dropout",
        )

        # === Decoder layers ===
        self.transformer_layers = []
        for i in range(num_layers):
            self.transformer_layers.append(
                keras_hub.layers.TransformerDecoder(
                    intermediate_dim=hidden_dim,
                    num_heads=num_heads,
                    dropout=dropout,
                    layer_norm_epsilon=1e-05,
                    # SASRec uses ReLU, although GeLU might be a better option
                    activation="relu",
                    kernel_initializer="glorot_uniform",
                    normalize_first=True,
                    dtype=dtype,
                    name=f"transformer_layer_{i}",
                )
            )

        # === Final layer norm ===
        self.layer_norm = keras.layers.LayerNormalization(
            axis=-1,
            epsilon=1e-8,
            dtype=dtype,
            name="layer_norm",
        )

        # === Retrieval ===
        # The layer that performs the retrieval.
        self.retrieval = keras_rs.layers.BruteForceRetrieval(k=k, return_scores=False)

        # === Loss ===
        self.loss_fn = keras.losses.BinaryCrossentropy(from_logits=True, reduction=None)

        # === Attributes ===
        self.vocabulary_size = vocabulary_size
        self.num_layers = num_layers
        self.num_heads = num_heads
        self.hidden_dim = hidden_dim
        self.dropout = dropout
        self.max_sequence_length = max_sequence_length

    def _get_last_non_padding_token(self, tensor, padding_mask):
        valid_token_mask = ops.logical_not(padding_mask)
        seq_lengths = ops.sum(ops.cast(valid_token_mask, "int32"), axis=1)
        last_token_indices = ops.maximum(seq_lengths - 1, 0)

        indices = ops.expand_dims(last_token_indices, axis=(-2, -1))
        gathered_tokens = ops.take_along_axis(tensor, indices, axis=1)
        last_token_embedding = ops.squeeze(gathered_tokens, axis=1)

        return last_token_embedding

    def build(self, input_shape):
        embedding_shape = list(input_shape) + [self.hidden_dim]

        # Model
        self.item_embedding.build(input_shape)
        self.position_embedding.build(embedding_shape)

        self.embeddings_add.build((embedding_shape, embedding_shape))
        self.embeddings_dropout.build(embedding_shape)

        for transformer_layer in self.transformer_layers:
            transformer_layer.build(decoder_sequence_shape=embedding_shape)

        self.layer_norm.build(embedding_shape)

        # Retrieval
        self.retrieval.candidate_embeddings = self.item_embedding.embeddings
        self.retrieval.build(input_shape)

        # Chain to super
        super().build(input_shape)

    def call(self, inputs, training=False):
        item_ids, padding_mask = inputs["item_ids"], inputs["padding_mask"]

        x = self.item_embedding(item_ids)
        position_embedding = self.position_embedding(x)
        x = self.embeddings_add((x, position_embedding))
        x = self.embeddings_dropout(x)

        for transformer_layer in self.transformer_layers:
            x = transformer_layer(x, decoder_padding_mask=padding_mask)

        item_sequence_embedding = self.layer_norm(x)
        result = {"item_sequence_embedding": item_sequence_embedding}

        # At inference, perform top-k retrieval.
        if not training:
            # need to extract last non-padding token.
            last_item_embedding = self._get_last_non_padding_token(
                item_sequence_embedding, padding_mask
            )
            result["predictions"] = self.retrieval(last_item_embedding)

        return result

    def compute_loss(self, x, y, y_pred, sample_weight, training=False):
        item_sequence_embedding = y_pred["item_sequence_embedding"]
        y_positive_sequence = y["positive_sequence"]
        y_negative_sequence = y["negative_sequence"]

        # Embed positive, negative sequences.
        positive_sequence_embedding = self.item_embedding(y_positive_sequence)
        negative_sequence_embedding = self.item_embedding(y_negative_sequence)

        # Logits
        positive_logits = ops.sum(
            ops.multiply(positive_sequence_embedding, item_sequence_embedding),
            axis=-1,
        )
        negative_logits = ops.sum(
            ops.multiply(negative_sequence_embedding, item_sequence_embedding),
            axis=-1,
        )
        logits = ops.concatenate([positive_logits, negative_logits], axis=1)

        # Labels
        labels = ops.concatenate(
            [
                ops.ones_like(positive_logits),
                ops.zeros_like(negative_logits),
            ],
            axis=1,
        )

        # sample weights
        sample_weight = ops.concatenate(
            [sample_weight, sample_weight],
            axis=1,
        )

        loss = self.loss_fn(
            y_true=ops.expand_dims(labels, axis=-1),
            y_pred=ops.expand_dims(logits, axis=-1),
            sample_weight=sample_weight,
        )
        loss = ops.divide_no_nan(ops.sum(loss), ops.sum(sample_weight))

        return loss

    def compute_output_shape(self, inputs_shape):
        return list(inputs_shape) + [self.hidden_dim]

In [ ]:
def early_stopping():
    return keras.callbacks.EarlyStopping(
        monitor='val_loss', 
        patience=15,
        restore_best_weights=True,
        verbose=1
    )

In [ ]:
itemnum = books.shape[0]
usernum = users.shape[0]

# SASREC + LLM

In [ ]:
class SasRecLLM(keras.Model):
    def __init__(
        self,
        vocabulary_size,
        num_layers,
        num_heads,
        hidden_dim,
        llm_embedding_matrix,
        dropout=0.0,
        max_sequence_length=100,
        dtype=None,
        k=20,
        **kwargs,
    ):
        super().__init__(dtype=dtype, **kwargs)
        
        self.llm_dim = llm_embedding_matrix.shape[1]

        # ======== Layers ========
        self.item_embedding = keras.layers.Embedding(
            input_dim=vocabulary_size,
            output_dim=self.llm_dim,
            embeddings_initializer=keras.initializers.Constant(llm_embedding_matrix),
            trainable=trainable,
            dtype=dtype,
            name="item_embedding",
        )
        
        # === 2. Projection Layer ===
        self.has_projection = self.llm_dim != hidden_dim
        if self.has_projection:
            self.proj_dense1 = keras.layers.Dense(units=hidden_dim * 2, activation='gelu', name="proj_dense1")
            self.proj_dropout = keras.layers.Dropout(dropout, name="proj_dropout")
            self.proj_dense2 = keras.layers.Dense(units=hidden_dim, activation=None, name="proj_dense2")
        else:
            self.projection = None
        

        # === 3. Position Embeddings ===
        self.position_embedding = keras_hub.layers.PositionEmbedding(
            initializer="glorot_uniform",
            sequence_length=max_sequence_length,
            dtype=dtype,
            name="position_embedding",
        )
        self.embeddings_add = keras.layers.Add(dtype=dtype, name="embeddings_add")
        self.embeddings_dropout = keras.layers.Dropout(dropout, dtype=dtype, name="embeddings_dropout")

        # === Decoder layers ===
        self.transformer_layers = []
        for i in range(num_layers):
            self.transformer_layers.append(
                keras_hub.layers.TransformerDecoder(
                    intermediate_dim=hidden_dim,
                    num_heads=num_heads,
                    dropout=dropout,
                    layer_norm_epsilon=1e-05,
                    activation="relu",
                    kernel_initializer="glorot_uniform",
                    normalize_first=True,
                    dtype=dtype,
                    name=f"transformer_layer_{i}",
                )
            )

        self.layer_norm = keras.layers.LayerNormalization(axis=-1, epsilon=1e-8, dtype=dtype, name="layer_norm")
        self.unit_norm = keras.layers.UnitNormalization(axis=-1)

        # === Loss ===
        self.loss_fn = keras.losses.BinaryCrossentropy(from_logits=True, reduction=None)

        # === Attributes ===
        self.vocabulary_size = vocabulary_size
        self.num_layers = num_layers
        self.num_heads = num_heads
        self.hidden_dim = hidden_dim
        self.dropout = dropout
        self.max_sequence_length = max_sequence_length

    def _get_last_non_padding_token(self, tensor, padding_mask):
        valid_token_mask = ops.logical_not(padding_mask)
        seq_lengths = ops.sum(ops.cast(valid_token_mask, "int32"), axis=1)
        last_token_indices = ops.maximum(seq_lengths - 1, 0)
        indices = ops.expand_dims(last_token_indices, axis=(-2, -1))
        gathered_tokens = ops.take_along_axis(tensor, indices, axis=1)
        return ops.squeeze(gathered_tokens, axis=1)

    def call(self, inputs, training=False):
        item_ids, padding_mask = inputs["item_ids"], inputs["padding_mask"]

        x = self.item_embedding(item_ids)
        
        if self.has_projection:
            x = self.proj_dense1(x)
            x = self.proj_dropout(x, training=training)
            x = self.proj_dense2(x)

        # scale embedding
        x = x * ops.sqrt(ops.cast(self.hidden_dim, dtype=x.dtype))
            
        position_embedding = self.position_embedding(x)
        x = self.embeddings_add((x, position_embedding))
        x = self.embeddings_dropout(x, training=training)

        seq_len = ops.shape(x)[1]
        causal_mask = ops.tril(ops.ones((seq_len, seq_len), dtype="bool"))
        
        for transformer_layer in self.transformer_layers:
            x = transformer_layer(x, decoder_padding_mask=padding_mask, training=training)

        item_sequence_embedding = self.layer_norm(x)
        result = {"item_sequence_embedding": item_sequence_embedding}

        # Inference
        if not training:
            last_item_embedding = self._get_last_non_padding_token(item_sequence_embedding, padding_mask)
            
            all_items = ops.arange(0, self.vocabulary_size)
            all_item_embs = self.item_embedding(all_items)
            
            if self.has_projection:
                all_item_embs = self.proj_dense1(all_item_embs)
                all_item_embs = self.proj_dropout(all_item_embs, training=False) # dropout is False for inference
                all_item_embs = self.proj_dense2(all_item_embs)
            
            # Normalize
            last_item_embedding = self.unit_norm(last_item_embedding)
            all_item_embs = self.unit_norm(all_item_embs)
                
            scores = ops.matmul(last_item_embedding, ops.transpose(all_item_embs))
            _, top_k_indices = ops.top_k(scores, k=20) 
            result["predictions"] = top_k_indices

        return result

    def compute_loss(self, x, y, y_pred, sample_weight, training=False):
        item_sequence_embedding = y_pred["item_sequence_embedding"]
        y_positive_sequence = y["positive_sequence"]
        y_negative_sequence = y["negative_sequence"]

        pos_emb = self.item_embedding(y_positive_sequence)
        neg_emb = self.item_embedding(y_negative_sequence)
        
        if self.has_projection:
            pos_emb = self.proj_dense2(self.proj_dropout(self.proj_dense1(pos_emb), training=training))
            neg_emb = self.proj_dense2(self.proj_dropout(self.proj_dense1(neg_emb), training=training))

        # Normalization
        pos_emb = self.unit_norm(pos_emb)
        neg_emb = self.unit_norm(neg_emb)
        item_sequence_embedding = self.unit_norm(item_sequence_embedding)

        # Logits
        positive_logits = ops.sum(ops.multiply(pos_emb, item_sequence_embedding), axis=-1)
        negative_logits = ops.sum(ops.multiply(neg_emb, item_sequence_embedding), axis=-1)
        logits = ops.concatenate([positive_logits, negative_logits], axis=1)

        # Labels & Weights
        labels = ops.concatenate([ops.ones_like(positive_logits), ops.zeros_like(negative_logits)], axis=1)
        sample_weight = ops.concatenate([sample_weight, sample_weight], axis=1)

        loss = self.loss_fn(
            y_true=ops.expand_dims(labels, axis=-1),
            y_pred=ops.expand_dims(logits, axis=-1),
            sample_weight=sample_weight,
        )
        return ops.divide_no_nan(ops.sum(loss), ops.sum(sample_weight))

In [ ]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 68.2 MB/s eta 0:00:00


In [ ]:
import faiss
import os

def load_index(index_nama):
    index_path = f"/kaggle/input/datasets/rahayukartikasari/upn-library-book-vector/{index_nama}_db.faiss"
    
    absolute_path = os.path.abspath(index_path)
    if not os.path.exists(index_path):
        raise FileNotFoundError(f"File tidak ditemukan!\nPython mencari di path: {absolute_path}\nPastikan nama folder dan file sudah benar.")
        
    index = faiss.read_index(index_path)
    vectors = index.reconstruct_n(0, index.ntotal)
    
    print(f"Berhasil mengambil {vectors.shape[0]} vektor dengan dimensi {vectors.shape[1]}")
    
    return vectors

In [ ]:
def evaluate(model, test_ds, k_list=K_TEST):
    if hasattr(model.item_embedding, 'embeddings'):
        item_embeddings = model.item_embedding.embeddings
    else:
        item_embeddings = model.item_embedding.get_weights()[0]
        
    if hasattr(model, 'has_projection') and model.has_projection:
        item_embeddings = model.proj_dense1(item_embeddings)
        item_embeddings = model.proj_dropout(item_embeddings, training=False)
        item_embeddings = model.proj_dense2(item_embeddings)

    item_embeddings = np.array(item_embeddings)
    clean_embeddings = np.nan_to_num(item_embeddings, nan=0.0, posinf=0.0, neginf=0.0)

    # Calculate candidates
    all_item_embs = model.item_embedding(ops.arange(0, model.vocabulary_size))
    if hasattr(model, 'has_projection') and model.has_projection:
        all_item_embs = model.proj_dense1(all_item_embs)
        all_item_embs = model.proj_dropout(all_item_embs, training=False)
        all_item_embs = model.proj_dense2(all_item_embs)
        
    # --- FIX 2: Apply the unit_norm to the items before transposing! ---
    if hasattr(model, 'unit_norm'):
        all_item_embs = model.unit_norm(all_item_embs)
        
    all_item_embs_transposed = np.array(ops.transpose(all_item_embs))

    results_raw = {k: {"hit": [], "ndcg": [], "unserendipity": []} for k in k_list}
    max_k = max(k_list)

    for batch in test_ds:
        inputs, labels = batch

        user_histories = inputs["item_ids"].numpy()
        padding_mask = inputs["padding_mask"].numpy()
        true_items = labels["positive_sequence"].numpy()

        outputs = model(inputs, training=False)
        user_states_raw = model._get_last_non_padding_token(outputs["item_sequence_embedding"], inputs["padding_mask"])
        
        if hasattr(model, 'unit_norm'):
            user_states_raw = model.unit_norm(user_states_raw)
            
        user_states = np.array(user_states_raw)
        
        all_scores = np.matmul(user_states, all_item_embs_transposed) 

        for i in range(len(true_items)):
            true_item = true_items[i]
            valid_history = user_histories[i][padding_mask[i]]

            user_scores = all_scores[i].copy()
            
            # Masking
            user_scores[valid_history] = -np.inf
            user_scores[0] = -np.inf 
            user_scores[true_item] = all_scores[i][true_item] 

            # Get Top-K highest scores
            top_k_rec = np.argsort(user_scores)[::-1][:max_k]

            for k in k_list:
                current_top_k = top_k_rec[:k]

                # --- Hit Ratio (HR@K) ---
                is_hit = 1.0 if true_item in current_top_k else 0.0
                results_raw[k]["hit"].append(is_hit)

                # --- NDCG@K ---
                if is_hit:
                    rank_idx = np.where(current_top_k == true_item)[0][0]
                    ndcg_val = 1.0 / np.log2(rank_idx + 2)
                else:
                    ndcg_val = 0.0
                results_raw[k]["ndcg"].append(ndcg_val)

                # --- unSerendipity@K ---
                if len(valid_history) > 0:
                    top_k_embs = clean_embeddings[current_top_k]
                    hist_embs = clean_embeddings[valid_history]
                    sim_scores = cosine_similarity(top_k_embs, hist_embs)
                    u_serendipity = np.mean(sim_scores)
                else:
                    u_serendipity = 0.0
                results_raw[k]["unserendipity"].append(u_serendipity)

    final_results = []
    for k in k_list:
        final_results.append({
            "k": k,
            "HitRate": np.mean(results_raw[k]["hit"]),
            "NDCG": np.mean(results_raw[k]["ndcg"]),
            "unSerendipity": np.mean(results_raw[k]["unserendipity"])
        })

    return final_results

# Grid Search

In [ ]:
def train_dynamic_model(model_name, h_dim, l, h, lr, trainable):
    keras.backend.clear_session()
    gc.collect()
    
    if model_name == "base_sasrec":
        model = SasRec(
            vocabulary_size=itemnum + 1,
            num_layers=l,
            num_heads=h,
            hidden_dim=h_dim,
            dropout=DROPOUT,
            max_sequence_length=MAX_CONTEXT_LENGTH,
            k=20,
        )
    else:
        emb = load_index(model_name)
        pad_vector = np.random.normal(scale=1e-5, size=(1, emb.shape[1]))
        emb_padded = np.vstack([pad_vector, emb])
        emb_padded = np.nan_to_num(emb_padded, nan=1e-5)
        
        model = SasRecLLM(
            vocabulary_size=itemnum + 1,
            num_layers=l,
            num_heads=h,
            hidden_dim=h_dim,
            llm_embedding_matrix=emb_padded,
            dropout=DROPOUT,
            max_sequence_length=MAX_CONTEXT_LENGTH,
            k=20,
            trainable=trainable,
        )
        
    # Build model with dummy batch
    dummy_batch = next(iter(train_ds.take(1)))
    model(dummy_batch[0], training=False)
    
    model.compile(optimizer=keras.optimizers.AdamW(learning_rate=lr))
    
    # Train
    print(f"\nTraining -> Model: {model_name.upper()} | Dim: {h_dim} | L: {l} | H: {h} | LR: {lr}")
    history = model.fit(
        x=train_ds,
        validation_data=val_ds,
        epochs=NUM_EPOCHS,
        callbacks=[early_stopping()],
        verbose=0 # Turn off epoch spam to keep notebook clean
    )
    
    return model

In [ ]:
for i, (h_dim, l, h, lr, model_name, trainable) in enumerate(all_combinations):
    print(f"\n{'='*40}")
    print(f"Run {i+1}/{len(all_combinations)}")
    print(f"{'='*40}")
    
    try:
        trained_model = train_dynamic_model(model_name, h_dim, l, h, lr, trainable)
        
        metrics = evaluate(trained_model, test_ds, k_list=K_TEST)
        
        for res in metrics:
            grid_results.append({
                "Model": model_name,
                "Hidden_Dim": h_dim,
                "Layers": l,
                "Heads": h,
                "LR": lr,
                "Trainable": trainable,
                "k": res["k"],
                "HitRate": res["HitRate"],
                "NDCG": res["NDCG"],
                "unSerendipity": res["unSerendipity"]
            })
            
        temp_df = pd.DataFrame(grid_results)
        temp_df.to_csv(csv_filename, index=False)
        print(f"✅ Kombinasi {i+1} selesai dan disimpan ke {csv_filename}")
        
    except Exception as e:
        print(f"❌ Error pada iterasi {i+1} (Dim:{h_dim}, L:{l}, H:{h}, LR:{lr}): {e}, Trainable: {trainable}")
        continue # If a model runs out of memory, skip it and keep going!

print("\n🎉 GRID SEARCH SELESAI!")
final_df = pd.DataFrame(grid_results)
final_df.to_csv("grid_search_final_complete.csv", index=False)


Run 1/324
Berhasil mengambil 14502 vektor dengan dimensi 256

Training -> Model: WORD2VEC | Dim: 256 | L: 1 | H: 1 | LR: 0.0001
Restoring model weights from the end of the best epoch: 192.
✅ Kombinasi 1 selesai dan disimpan ke grid_search_results_live.csv

Run 2/324
Berhasil mengambil 14502 vektor dengan dimensi 256

Training -> Model: WORD2VEC | Dim: 256 | L: 1 | H: 1 | LR: 0.0001
Epoch 16: early stopping
Restoring model weights from the end of the best epoch: 1.
✅ Kombinasi 2 selesai dan disimpan ke grid_search_results_live.csv

Run 3/324
Berhasil mengambil 14502 vektor dengan dimensi 768

Training -> Model: BERT_MULTI | Dim: 256 | L: 1 | H: 1 | LR: 0.0001
Epoch 54: early stopping
Restoring model weights from the end of the best epoch: 39.
✅ Kombinasi 3 selesai dan disimpan ke grid_search_results_live.csv

Run 4/324
Berhasil mengambil 14502 vektor dengan dimensi 768

Training -> Model: BERT_MULTI | Dim: 256 | L: 1 | H: 1 | LR: 0.0001
Epoch 16: early stopping
Restoring model weights 

# RESULT

In [1]:
import pandas as pd

df = pd.read_csv("results/grid_search_final_complete.csv")

df_10 = df[df['k'] == 10]

best_models = df_10.sort_values(by=['NDCG', 'HitRate'], ascending=[False, False])

columns_to_show = ['Model', 'Hidden_Dim', 'Layers', 'Heads', 'LR', 'Trainable', 'HitRate', 'NDCG', 'unSerendipity']
print("🏆 TOP 10 MODEL TERBAIK (Berdasarkan NDCG@10) 🏆")
display(best_models[columns_to_show].head(10))

🏆 TOP 10 MODEL TERBAIK (Berdasarkan NDCG@10) 🏆


,Model,Hidden_Dim,Layers,Heads,LR,Trainable,HitRate,NDCG,unSerendipity
512,qwen,1024,2,1,0.00010,True,0.243286,0.177755,0.606096
632,qwen,1024,3,3,0.00010,True,0.230648,0.170525,0.599614
536,qwen,1024,2,2,0.00010,True,0.233807,0.159937,0.591599
464,qwen,1024,1,2,0.00010,True,0.227488,0.157749,0.611075
608,qwen,1024,3,2,0.00010,True,0.218009,0.156980,0.578116
440,qwen,1024,1,1,0.00010,True,0.219589,0.153308,0.614335
488,qwen,1024,1,3,0.00010,True,0.219589,0.153295,0.602654
584,qwen,1024,3,1,0.00010,True,0.206951,0.146838,0.583757
560,qwen,1024,2,3,0.00010,True,0.200632,0.140622,0.560929
596,qwen,1024,3,1,0.00001,True,0.173776,0.116345,0.608139


In [2]:
bert_model_10 = df[(df['Model'] == 'bert_multi') & (df['k'] == 10)]
best_bert_model = bert_model_10.sort_values(by=['NDCG', 'HitRate'], ascending=[False, False])
best_bert_model.head(10)

,Model,Hidden_Dim,Layers,Heads,LR,Trainable,k,HitRate,NDCG,unSerendipity
450,bert_multi,1024,1,1,0.00001,False,10,0.050553,0.036746,0.820082
474,bert_multi,1024,1,2,0.00001,False,10,0.041074,0.036473,0.782511
498,bert_multi,1024,1,3,0.00001,False,10,0.044234,0.035428,0.796686
486,bert_multi,1024,1,3,0.00010,False,10,0.041074,0.034369,0.804468
16,bert_multi,256,1,1,0.00001,True,10,0.056872,0.033224,0.861212
438,bert_multi,1024,1,1,0.00010,False,10,0.042654,0.032972,0.851250
582,bert_multi,1024,3,1,0.00010,False,10,0.044234,0.032784,0.838289
570,bert_multi,1024,2,3,0.00001,False,10,0.044234,0.032336,0.844319
268,bert_multi,512,1,3,0.00010,True,10,0.053712,0.031948,0.837459
244,bert_multi,512,1,2,0.00010,True,10,0.048973,0.031749,0.843322


In [3]:
w2v_model_10 = df[(df['Model'] == 'word2vec') & (df['k'] == 10)]
best_w2v_model = w2v_model_10.sort_values(by=['NDCG', 'HitRate'], ascending=[False, False])
best_w2v_model.head(10)

,Model,Hidden_Dim,Layers,Heads,LR,Trainable,k,HitRate,NDCG,unSerendipity
96,word2vec,256,2,2,0.0001,True,10,0.093207,0.053627,0.730309
48,word2vec,256,1,3,0.0001,True,10,0.088468,0.048483,0.726838
120,word2vec,256,2,3,0.0001,True,10,0.090047,0.046350,0.726650
0,word2vec,256,1,1,0.0001,True,10,0.085308,0.046105,0.724753
72,word2vec,256,2,1,0.0001,True,10,0.085308,0.045664,0.722902
144,word2vec,256,3,1,0.0001,True,10,0.083728,0.045598,0.718535
24,word2vec,256,1,2,0.0001,True,10,0.082148,0.044766,0.724532
168,word2vec,256,3,2,0.0001,True,10,0.078989,0.043178,0.730051
192,word2vec,256,3,3,0.0001,True,10,0.086888,0.041774,0.728329
264,word2vec,512,1,3,0.0001,True,10,0.060032,0.035834,0.855529


In [5]:
average_df = df_10.groupby(['Trainable', 'Hidden_Dim', 'Layers', 'Heads', 'LR']).agg(
    # Max_NDCG=('NDCG', 'max'),
    Avg_NDCG=('NDCG', 'mean'),
    # Max_HitRate=('HitRate', 'max'),
    Avg_HitRate=('HitRate', 'mean'),
    Avg_unSerendipity=('unSerendipity', 'mean')
).reset_index()

average_df = average_df.sort_values(by='Avg_NDCG', ascending=False)
average_df.head(10)

,Trainable,Hidden_Dim,Layers,Heads,LR,Avg_NDCG,Avg_HitRate,Avg_unSerendipity
97,True,1024,2,1,0.00010,0.071897,0.105845,0.750719
107,True,1024,3,3,0.00010,0.069261,0.101632,0.744038
95,True,1024,1,3,0.00010,0.068123,0.103739,0.758978
91,True,1024,1,1,0.00010,0.064508,0.097420,0.766556
99,True,1024,2,2,0.00010,0.064050,0.098999,0.749788
105,True,1024,3,2,0.00010,0.063664,0.095313,0.740204
93,True,1024,1,2,0.00010,0.059795,0.090047,0.749824
103,True,1024,3,1,0.00010,0.058831,0.087414,0.743246
101,True,1024,2,3,0.00010,0.057047,0.086361,0.725724
90,True,1024,1,1,0.00001,0.053193,0.085308,0.763648
